No,7

In [2]:
from pathlib import Path


# --- 共通の復元基盤 undo_utils.py を読み込む ---
# ※ このセルの import を並べ替えても壊れないように、undo_utils だけは
#    import 文ではなく importlib 経由で読み込んでいます。
import importlib
import sys
from pathlib import Path


def _locate_undo_utils():
    """undo_utils.py があるフォルダを探す（VS Code の作業ディレクトリ設定に依存しない）。"""
    candidates = []
    # 1) VS Code がノートブック自身のパスを教えてくれる場合
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        candidates.append(Path(nb_file).parent)
    # 2) 作業ディレクトリと、その中／親の「コードフォルダ」
    cwd = Path.cwd()
    candidates += [cwd, cwd / "コードフォルダ", cwd.parent, cwd.parent / "コードフォルダ"]

    for c in candidates:
        if (c / "undo_utils.py").is_file():
            return c.resolve()

    raise FileNotFoundError(
        "undo_utils.py が見つかりません。\n"
        "このノートブックと同じ「コードフォルダ」内に undo_utils.py があるか確認してください。\n"
        f"探した場所: {[str(c) for c in candidates]}"
    )


_uu_dir = str(_locate_undo_utils())
if _uu_dir not in sys.path:
    sys.path.insert(0, _uu_dir)

uu = importlib.import_module("undo_utils")
importlib.reload(uu)  # undo_utils.py を編集した場合も反映されるようにする

<module 'undo_utils' from 'C:\\Users\\0uh2j\\Desktop\\vscodeで\\ファイル整理２\\コードフォルダ\\undo_utils.py'>

メインフォルダを選ぶと、各**末端フォルダ**（サブフォルダを1つも持たないフォルダ）に入っている
すべてのファイルを**ひとつ前（親）のフォルダ**へ移動し、空になった末端フォルダを削除します。

```
処理前                                処理後
  メイン/サブ/サブサブ/データ/a.csv  →  メイン/サブ/サブサブ/a.csv
                    （データフォルダは削除）
```

**⚠️ 実行前に必ず読んでください**

- セル **`[4]` → `[6]` の順に、同じセッションで続けて実行**してください。
  `[6]` は `[4]` が用意した対象リストを使います（02 の `[3]`→`[5]` と同じ関係）。
  カーネルを再起動した場合は `[4]` からやり直してください。
- **1回の実行で1階層だけ浅くなります。** もう一度実行すると、さらにもう1階層持ち上がります
  （元の末端フォルダが消えて、その親が新しい末端フォルダになるため）。
  意図しない限り繰り返し実行しないでください。
- 移動先に同名ファイルがある場合は、`data.csv` → `data_末端フォルダ名.csv` のように
  **末端フォルダ名を付けて**区別します。上書きは絶対にしません。
- 隠しファイル（`.DS_Store` など）は移動しません。そのため、それが残るフォルダは
  **削除されず保持**されます。
- 末端フォルダがメインフォルダの直下にある場合、ファイルはメインフォルダ直下へ移動します。

### 【1工程目】対象の末端フォルダを確認する

このセルは**ファイルを一切変更しません**。何が対象になるかを表示するだけです。
内容を確認してから、次の実行セルに進んでください。

In [4]:
# === 【1工程目】対象の末端フォルダを確認する（ファイルは変更しません） ===

main_folder_path = uu.select_folder("メインフォルダを選択してください")

terminal_folders = []

if main_folder_path is not None:

    def target_files(folder):
        """そのフォルダ直下の加工対象ファイル（隠しファイル以外）。"""
        return [p for p in sorted(folder.iterdir())
                if p.is_file() and not p.name.startswith(".")]

    # 末端フォルダ = サブフォルダを1つも持たないフォルダ
    #   uu.iter_dirs は _undo / _trash を除外して再帰的に走査する
    leaf_folders = [d for d in uu.iter_dirs(main_folder_path)
                    if not any(c.is_dir() for c in d.iterdir())]

    # 加工対象ファイルが1つ以上入っているものだけを対象にする
    terminal_folders = [d for d in leaf_folders if target_files(d)]

    if not terminal_folders:
        print("移動対象のファイルが入った末端フォルダは見つかりませんでした。")
    else:
        print(f"対象の末端フォルダ: {len(terminal_folders)} 個")
        print("=" * 70)

        total_files = 0
        renamed = 0

        # 移動先ごとに「埋まる予定の名前」を積み上げていく。
        # 同じ親フォルダを共有する兄弟の末端フォルダどうしの衝突も拾うため、
        # 末端フォルダをまたいで持ち越すのが要点（実行セルと同じ順序で処理する）。
        claimed = {}

        for terminal in terminal_folders:
            files = target_files(terminal)
            total_files += len(files)

            parent = terminal.parent
            if parent not in claimed:
                claimed[parent] = {p.name for p in parent.iterdir()}

            rel = terminal.relative_to(main_folder_path)
            parent_rel = "（メインフォルダ直下）" if parent == main_folder_path \
                else parent.relative_to(main_folder_path)

            print(f"■ {rel}")
            print(f"    対象ファイル {len(files)} 件  ->  移動先: {parent_rel}")

            # 移動先で名前が衝突するものを予告する（実行セルと同じ規則）
            for f in files:
                if f.name in claimed[parent]:
                    new_name = f"{f.stem}_{terminal.name}{f.suffix}"
                    n = 1
                    while new_name in claimed[parent]:
                        new_name = f"{f.stem}_{terminal.name}__{n}{f.suffix}"
                        n += 1
                    print(f"      ※ {f.name} は {new_name} にリネームされます")
                    claimed[parent].add(new_name)
                    renamed += 1
                else:
                    claimed[parent].add(f.name)

            # 隠しファイルが残る場合はフォルダが保持される
            hidden = [p for p in terminal.iterdir()
                      if p.is_file() and p.name.startswith(".")]
            if hidden:
                names = ", ".join(p.name for p in hidden)
                print(f"      ※ {names} が残るため、このフォルダは削除されず保持されます")

        print("=" * 70)
        print(f"合計: 末端フォルダ {len(terminal_folders)} 個 / "
              f"移動予定ファイル {total_files} 件 / リネーム {renamed} 件")
        print()
        print("内容を確認したら、次のセルを実行してください。")

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-直接式樹脂圧力本実験データ再々
対象の末端フォルダ: 102 個
■ 1.0mm\190℃\010\2026_0805_130808_345
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\010
■ 1.0mm\190℃\020\2026_0805_132154_274
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\020
■ 1.0mm\190℃\040\2026_0805_133431_638
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\040
■ 1.0mm\190℃\080\2026_0805_134232_860
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\080
■ 1.0mm\190℃\120\2026_0805_135110_705
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\120
■ 1.0mm\190℃\160\2026_0805_140315_001
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\160
■ 1.0mm\190℃\200\2026_0805_141053_893
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\200
■ 1.0mm\190℃\240\2026_0805_141929_038
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\240
■ 1.0mm\190℃\280\2026_0805_143553_374
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\280
■ 1.0mm\190℃\320\2026_0805_144352_881
    対象ファイル 26 件  ->  移動先: 1.0mm\190℃\320
■ 1.0mm\210℃\010\2026_0805_153832_110
    対象ファイル 28 件  ->  移動先: 1.0mm\210℃\

### 【2工程目】移動と末端フォルダの削除を実行する

1工程目で表示された内容でよければ、このセルを実行します。
実行内容は記録されるので、末尾の復元セルで元に戻せます。

In [5]:
# === 【2工程目】ファイルをひとつ前のフォルダへ移動し、末端フォルダを削除 ===
# 実行内容は _undo/undo_log.json に記録され、末尾の復元セルで元に戻せます。

STEP_NAME = "07_末端フォルダのファイルをひとつ前のフォルダへ移動"

if "terminal_folders" not in globals() or "main_folder_path" not in globals():
    print("エラー: 対象リストが見つかりません。先に1工程目のセルを実行してください。")
elif main_folder_path is None:
    print("エラー: メインフォルダが選択されていません。先に1工程目のセルを実行してください。")
elif not terminal_folders:
    print("対象の末端フォルダがありません。先に1工程目のセルを実行して確認してください。")
else:
    moved = 0
    deleted = 0
    kept = 0

    with uu.UndoJournal(main_folder_path, STEP_NAME) as j:
        # 1工程目で確定したリストをそのまま使う。
        # ここで末端フォルダを列挙し直すと、フォルダを消したことで親が新たな末端に
        # なってしまい、際限なく階層が潰れてしまう。
        for terminal in terminal_folders:
            if not terminal.is_dir():
                print(f"スキップ: {terminal} は既に存在しません")
                continue

            parent = terminal.parent
            files = [p for p in sorted(terminal.iterdir())
                     if p.is_file() and not p.name.startswith(".")]

            for f in files:
                dest = parent / f.name

                # 同名ファイルがある場合は末端フォルダ名を付けて区別する（上書きしない）
                if dest.exists():
                    dest = parent / f"{f.stem}_{terminal.name}{f.suffix}"
                    n = 1
                    while dest.exists():
                        dest = parent / f"{f.stem}_{terminal.name}__{n}{f.suffix}"
                        n += 1
                    print(f"  名前の衝突を回避: {f.name} -> {dest.name}")

                try:
                    j.move(f, dest)
                    moved += 1
                except Exception as e:
                    print(f"  移動エラー ({f.name}): {e}")

            # 空になっていれば削除。隠しファイル等が残っていれば保持する。
            rel = terminal.relative_to(main_folder_path)
            if j.rmdir(terminal):
                print(f"削除: {rel}")
                deleted += 1
            else:
                print(f"保持: {rel}（対象外のファイルが残っています）")
                kept += 1

    print("-" * 70)
    print(f"移動したファイル: {moved} 件")
    print(f"削除した末端フォルダ: {deleted} 個 / 保持した末端フォルダ: {kept} 個")

削除: 1.0mm\190℃\010\2026_0805_130808_345
削除: 1.0mm\190℃\020\2026_0805_132154_274
削除: 1.0mm\190℃\040\2026_0805_133431_638
削除: 1.0mm\190℃\080\2026_0805_134232_860
削除: 1.0mm\190℃\120\2026_0805_135110_705
削除: 1.0mm\190℃\160\2026_0805_140315_001
削除: 1.0mm\190℃\200\2026_0805_141053_893
削除: 1.0mm\190℃\240\2026_0805_141929_038
削除: 1.0mm\190℃\280\2026_0805_143553_374
削除: 1.0mm\190℃\320\2026_0805_144352_881
削除: 1.0mm\210℃\010\2026_0805_153832_110
削除: 1.0mm\210℃\020\2026_0805_154725_048
削除: 1.0mm\210℃\040\2026_0805_155620_504
削除: 1.0mm\210℃\080\2026_0805_160510_683
削除: 1.0mm\210℃\120\2026_0805_161554_199
削除: 1.0mm\210℃\160\2026_0805_162505_532
削除: 1.0mm\210℃\200\2026_0805_163319_759
削除: 1.0mm\210℃\240\2026_0805_164306_541
削除: 1.0mm\210℃\280\2026_0805_165522_094
削除: 1.0mm\210℃\320\2026_0805_170720_940
削除: 1.0mm\230℃\010\2026_0805_174625_673
削除: 1.0mm\230℃\020\2026_0805_180554_414
削除: 1.0mm\230℃\040\2026_0805_181408_718
削除: 1.0mm\230℃\080\2026_0805_182309_112
削除: 1.0mm\230℃\120\2026_0805_183223_175


---
### 🔎 現在のフォルダ構成を調べる（確認用・いつでも実行できます）

このセルは**ファイルを一切変更しません**。次の3つを表示します。

1. 現在のフォルダ構成（メインフォルダからの深さごとの集計）
2. **02 を実行したらどうなるか**の予測（実際には動かさずシミュレートします）
3. その後 04・05 が付けるファイル名の予測

工程の順序を入れ替えたとき（例: 01 を飛ばして 07 を先に実行したとき）に、
02 のセル `[7]` や 04 の命名が意図どおりになるかを、実行前に確かめるためのものです。

In [6]:
# === 🔎 現在のフォルダ構成を調べる（ファイルは変更しません） ===
# 02 を実行した場合の結果をシミュレートして表示します。実際には何も動かしません。

from collections import defaultdict

SAMPLE = 3  # 各項目で表示するサンプル件数


def diagnose():
    root = uu.select_folder("調べたいメインフォルダを選択してください")
    if root is None:
        return

    # ---------------------------------------------------------------
    # 現在の状態を「メインフォルダからの相対パス」の集合として取り込む
    # ---------------------------------------------------------------
    files = []   # 可視ファイル（_undo / _trash と隠しファイルは除外）
    dirs = []
    for p in sorted(root.rglob("*")):
        if uu.is_reserved(p, root):
            continue
        rel = p.relative_to(root)
        if p.is_dir():
            dirs.append(rel)
        elif not p.name.startswith("."):
            files.append(rel)

    if not files:
        print("可視ファイルが1件も見つかりませんでした。")
        return

    # ---------------------------------------------------------------
    # 1. 現在の構成を深さ別に集計
    # ---------------------------------------------------------------
    print("=" * 72)
    print("【1】現在のフォルダ構成")
    print("=" * 72)
    print(f"メインフォルダ: {root}")
    print()

    dirs_by_depth = defaultdict(list)
    for d in dirs:
        dirs_by_depth[len(d.parts)].append(d)

    files_by_depth = defaultdict(list)
    for f in files:
        # ファイルが入っているフォルダの深さ（メインフォルダ直下 = 0）
        files_by_depth[len(f.parts) - 1].append(f)

    for depth in sorted(set(dirs_by_depth) | set(files_by_depth)):
        nd = len(dirs_by_depth.get(depth, []))
        nf = len(files_by_depth.get(depth, []))
        label = "メインフォルダ直下" if depth == 0 else f"深さ {depth}"
        print(f"  {label}: フォルダ {nd} 個 / ファイル {nf} 件")
        for d in dirs_by_depth.get(depth, [])[:SAMPLE]:
            print(f"      └ {d}")

    print()
    depths = sorted(files_by_depth)
    if len(depths) == 1:
        d = depths[0]
        print(f"  ファイルはすべて深さ {d} にあります"
              f"（メイン/{'/'.join(['□'] * d)}/*.csv）")
    else:
        print(f"  ⚠ ファイルの深さが揃っていません: {depths}")
        print("     02 のセル [7] や 04 の命名は深さを前提にしているため、")
        print("     このまま進めると一部だけ意図しない結果になる可能性があります。")

    # ---------------------------------------------------------------
    # 2. 02 を実行した場合をシミュレート（ディスクには触れない）
    # ---------------------------------------------------------------
    print()
    print("=" * 72)
    print("【2】02 を実行した場合の予測（シミュレーション）")
    print("=" * 72)

    # --- セル [3]: 拡張子ごとの仕分け ---
    #     メイン/<拡張子>/<現在の親の相対パス>/<ファイル名>
    after3 = []
    for f in files:
        ext = f.suffix
        if not ext:
            continue  # 拡張子が無いファイルは触らない
        parent = f.parent
        base = Path(ext[1:])
        after3.append(base / f.name if str(parent) == "."
                      else base / parent / f.name)

    print(f"  [3] 仕分け        : {len(after3)} 件のファイルを移動")
    for p in after3[:SAMPLE]:
        print(f"      └ {p}")

    # --- セル [5]: 可視ファイルを含まなくなった直下フォルダを削除 ---
    top_with_files = {p.parts[0] for p in after3}
    removed_tops = sorted({d.parts[0] for d in dirs} - top_with_files)
    print(f"  [5] 空フォルダ削除: 直下の {len(removed_tops)} 個のフォルダツリーを削除")
    for t in removed_tops[:SAMPLE]:
        print(f"      └ {t}")

    # --- セル [7]: root/サブ/サブサブ/データフォルダ の3段を辿って移動 ---
    #     実コードは各階層を sorted() で先に確定させてから移動するので、
    #     ここでも同じ順序で「移動対象」を先に決めてから適用する。
    dirset = {p.parent for p in after3}
    for p in list(dirset):                      # 途中の階層も補う
        cur = p
        while cur != Path("."):
            dirset.add(cur)
            cur = cur.parent
    dirset.discard(Path("."))

    def children(d):
        return sorted({x for x in dirset if x.parent == d})

    moves = []                                   # (移動元, 移動先)
    for sub_dir in children(Path(".")):
        for sub_sub_dir in children(sub_dir):
            ext_name = sub_sub_dir.name          # サブサブ名を拡張子名として扱う
            for data_folder in children(sub_sub_dir):
                new_parent = Path(ext_name) / sub_dir.name
                moves.append((data_folder, new_parent / data_folder.name))

    print(f"  [7] 最終段階      : {len(moves)} 個のフォルダを移動")
    if not moves:
        print("      ⚠ セル [7] は何も移動しません。")
        print("         このセルは メイン/サブ/サブサブ/データフォルダ の3段を前提に")
        print("         しています。階層が浅いと対象が見つからず、実行しても")
        print("         何も起きません（害はありませんが、意味もありません）。")
    else:
        for src, dst in moves[:SAMPLE]:
            print(f"      └ {src}  ->  {dst}")

    # 最終的なファイル配置を求める
    final = []
    for p in after3:
        for src, dst in moves:
            if p.is_relative_to(src):
                p = dst / p.relative_to(src)
                break
        final.append(p)

    print()
    print("  02 をすべて実行したあとのファイル配置（予測）:")
    for p in final[:SAMPLE]:
        print(f"      └ {p}")
    fdepths = sorted({len(p.parts) - 1 for p in final})
    print(f"  ファイルの深さ: {fdepths}")

    # ---------------------------------------------------------------
    # 3. 04・05 が付けるファイル名の予測
    # ---------------------------------------------------------------
    print()
    print("=" * 72)
    print("【3】そのあと 04・05 が付けるファイル名の予測")
    print("=" * 72)

    sample = next((p for p in final if len(p.parts) >= 3), None)
    if sample is None:
        print("  ⚠ 階層が浅すぎて、04 の命名規則（親・祖父フォルダ名を使う）が")
        print("     成り立ちません。")
        return

    terminal = sample.parent.name          # 末端フォルダ名
    pre_terminal = sample.parent.parent.name   # そのひとつ前のフォルダ名
    print(f"  例にするファイル: {sample}")
    print(f"    末端フォルダ名          = {terminal}")
    print(f"    そのひとつ前のフォルダ名 = {pre_terminal}")
    print()
    print(f"  04 が付ける名前: 0<連番>_{pre_terminal}_{terminal}.csv")
    print(f"  05 が付ける名前: <元の名前>_{pre_terminal}.csv")
    print()
    if "csv" in (terminal, pre_terminal):
        print("  ⚠ 拡張子フォルダ 'csv' がファイル名に混ざります。")
        print("     意図した命名（例: 0123_160_条件名.csv）にならない可能性が高いです。")
        print("     02 のセル [7] を実行しない、という選択も検討してください。")
    else:
        print("  上の2つの名前が意図どおりであれば、このまま進めて問題ありません。")


diagnose()

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-直接式樹脂圧力本実験データ再々
【1】現在のフォルダ構成
メインフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\202603-4 - 本実験圧力データ\202606・08-実験データ再々\202606・08-直接式樹脂圧力本実験データ再々

  深さ 1: フォルダ 3 個 / ファイル 0 件
      └ 1.0mm
      └ 1.5mm
      └ 2.0mm
  深さ 2: フォルダ 9 個 / ファイル 0 件
      └ 1.0mm\190℃
      └ 1.0mm\210℃
      └ 1.0mm\230℃
  深さ 3: フォルダ 102 個 / ファイル 2602 件
      └ 1.0mm\190℃\010
      └ 1.0mm\190℃\020
      └ 1.0mm\190℃\040

  ファイルはすべて深さ 3 にあります（メイン/□/□/□/*.csv）

【2】02 を実行した場合の予測（シミュレーション）
  [3] 仕分け        : 2602 件のファイルを移動
      └ csv\1.0mm\190℃\010\010$0.csv
      └ xdt\1.0mm\190℃\010\010$0.xdt
      └ csv\1.0mm\190℃\010\010$1.csv
  [5] 空フォルダ削除: 直下の 3 個のフォルダツリーを削除
      └ 1.0mm
      └ 1.5mm
      └ 2.0mm
  [7] 最終段階      : 18 個のフォルダを移動
      └ csv\1.0mm\190℃  ->  1.0mm\csv\190℃
      └ csv\1.0mm\210℃  ->  1.0mm\csv\210℃
      └ csv\1.0mm\230℃  ->  1.0mm\csv\230℃

  02 をすべて実行したあとのファイル配置（予測）:
      └ 1.0mm\csv\

---
### ⏪ 復元（元に戻す）


このセルを実行すると、**このノートブックで行った直前の1工程**を巻き戻します。
（メインフォルダの `_undo/undo_log.json` に記録された履歴を使います）


繰り返し実行すれば、01〜06 のどの工程まででもさかのぼれます。
削除したファイルは `_trash` フォルダに退避されているので、これも一緒に元の場所へ戻ります。

In [ ]:
# ===== 共通の復元セル =====
# 直前に実行した1工程を巻き戻します。
# 続けて実行すれば、さらに1つ前の工程へとさかのぼれます。

uu.undo_interactive()